<left>
    <img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" width="20%">
</left>

# Demo: Human-in-the-Loop Control

**Purpose:** Demonstrate a real approval boundary: propose an action, save a checkpoint, then resume after a human decision.

**Scenario:** A legal document assistant may draft a recommendation, but sending it to a client requires explicit approval.

**Learning outcomes**

- Distinguish a proposed action from an executed action.
- Persist the pending action before asking for approval.
- Resume the same checkpoint through approve and reject paths.
- Verify side effects rather than trusting the final text.

## 1. Setup

In [1]:
# Run this cell first. The key is requested securely and is never printed.
import importlib.util
import os
import subprocess
import sys
from getpass import getpass

if importlib.util.find_spec("openai") is None or importlib.util.find_spec("socksio") is None:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "openai==1.109.1",
        "socksio==1.0.0",
    ])

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

from openai import OpenAI

MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
client = OpenAI(timeout=30.0, max_retries=2)
print(f"Setup complete. Model: {MODEL}")

Setup complete. Model: gpt-4o-mini


## 2. Proposed action and simulated side effect

In [2]:
import json
from copy import deepcopy

SIDE_EFFECTS = []

def send_contract_recommendation(email: str, recommendation: str) -> dict:
    """Simulated external send. This function must run only after approval."""
    event = {
        "tool": "send_contract_recommendation",
        "email": email,
        "recommendation": recommendation,
    }
    SIDE_EFFECTS.append(event)
    return {"sent": True, **event}

SEND_TOOL = {
        "type": "function",
        "function": {
            "name": "send_contract_recommendation",
            "description": "Send a legal recommendation to a client. This action requires human approval.",
            "parameters": {
                "type": "object",
                "properties": {"email": {"type": "string"}, "recommendation": {"type": "string"}},
                "required": ["email", "recommendation"],
                "additionalProperties": False
            }
        }
    }

def create_approval_checkpoint(user_request: str) -> dict:
    """Ask the model to propose—not execute—the high-risk action."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "Convert the user's request into a send_contract_recommendation tool call. "
                    "Do not claim that it was sent. A separate human approval step controls execution."
                ),
            },
            {"role": "user", "content": user_request},
        ],
        tools=[SEND_TOOL],
        tool_choice={"type": "function", "function": {"name": "send_contract_recommendation"}},
        temperature=0,
    )
    call = response.choices[0].message.tool_calls[0]
    arguments = json.loads(call.function.arguments)
    return {
        "status": "awaiting_approval",
        "pending_action": {
            "tool": call.function.name,
            "arguments": arguments,
        },
        "audit_log": [{
            "event": "approval_requested",
            "tool": call.function.name,
            "arguments": arguments,
        }],
    }

print("Approval workflow ready")

Approval workflow ready


## 3. Approval policy and resume logic

In [3]:
APPROVAL_POLICY = {
    "send_contract_recommendation": "REQUIRE_APPROVAL"
}

def resume_after_approval(checkpoint: dict, approved: bool, reviewer: str) -> dict:
    state = deepcopy(checkpoint)
    if state.get("status") != "awaiting_approval":
        raise ValueError("Checkpoint is not awaiting approval.")

    action = state["pending_action"]
    policy = APPROVAL_POLICY.get(action["tool"], "DENY")

    if policy != "REQUIRE_APPROVAL":
        state["status"] = "blocked"
        state["audit_log"].append({
            "event": "blocked",
            "reviewer": reviewer,
            "reason": "No approval policy exists for this action.",
        })
        return state

    if not approved:
        state["status"] = "rejected"
        state["audit_log"].append({
            "event": "approval_rejected",
            "reviewer": reviewer,
        })
        return state

    result = send_contract_recommendation(**action["arguments"])
    state["status"] = "executed"
    state["result"] = result
    state["audit_log"].append({
        "event": "approval_granted",
        "reviewer": reviewer,
    })
    state["audit_log"].append({
        "event": "action_executed",
        "tool": action["tool"],
        "result": result,
    })
    return state

print("Approval policy ready")

Approval policy ready


## 4. Create and serialize the checkpoint

In [4]:
SIDE_EFFECTS.clear()

checkpoint = create_approval_checkpoint(
    "Send a recommendation to client@company.com saying that the agreement should be reviewed by counsel before signing."
)

# A JSON round-trip demonstrates that the pending state can be persisted and restored.
checkpoint_json = json.dumps(checkpoint)
restored_checkpoint = json.loads(checkpoint_json)

print(json.dumps(restored_checkpoint, indent=2))
assert restored_checkpoint["status"] == "awaiting_approval"
assert SIDE_EFFECTS == []
print("\nVerification passed: proposing the action did not execute it.")

{
  "status": "awaiting_approval",
  "pending_action": {
    "tool": "send_contract_recommendation",
    "arguments": {
      "email": "client@company.com",
      "recommendation": "The agreement should be reviewed by counsel before signing."
    }
  },
  "audit_log": [
    {
      "event": "approval_requested",
      "tool": "send_contract_recommendation",
      "arguments": {
        "email": "client@company.com",
        "recommendation": "The agreement should be reviewed by counsel before signing."
      }
    }
  ]
}

Verification passed: proposing the action did not execute it.


## 5. Reject path

In [5]:
SIDE_EFFECTS.clear()
rejected_state = resume_after_approval(
    restored_checkpoint,
    approved=False,
    reviewer="legal-reviewer@example.com",
)

print(json.dumps(rejected_state, indent=2))
assert rejected_state["status"] == "rejected"
assert SIDE_EFFECTS == []
print("\nVerification passed: rejection produced no side effect.")

{
  "status": "rejected",
  "pending_action": {
    "tool": "send_contract_recommendation",
    "arguments": {
      "email": "client@company.com",
      "recommendation": "The agreement should be reviewed by counsel before signing."
    }
  },
  "audit_log": [
    {
      "event": "approval_requested",
      "tool": "send_contract_recommendation",
      "arguments": {
        "email": "client@company.com",
        "recommendation": "The agreement should be reviewed by counsel before signing."
      }
    },
    {
      "event": "approval_rejected",
      "reviewer": "legal-reviewer@example.com"
    }
  ]
}

Verification passed: rejection produced no side effect.


## 6. Approve path

In [6]:
SIDE_EFFECTS.clear()
approved_state = resume_after_approval(
    restored_checkpoint,
    approved=True,
    reviewer="legal-reviewer@example.com",
)

print(json.dumps(approved_state, indent=2))
assert approved_state["status"] == "executed"
assert approved_state["result"]["sent"] is True
assert len(SIDE_EFFECTS) == 1
print("\nVerification passed: the action executed exactly once after approval.")

{
  "status": "executed",
  "pending_action": {
    "tool": "send_contract_recommendation",
    "arguments": {
      "email": "client@company.com",
      "recommendation": "The agreement should be reviewed by counsel before signing."
    }
  },
  "audit_log": [
    {
      "event": "approval_requested",
      "tool": "send_contract_recommendation",
      "arguments": {
        "email": "client@company.com",
        "recommendation": "The agreement should be reviewed by counsel before signing."
      }
    },
    {
      "event": "approval_granted",
      "reviewer": "legal-reviewer@example.com"
    },
    {
      "event": "action_executed",
      "tool": "send_contract_recommendation",
      "result": {
        "sent": true,
        "tool": "send_contract_recommendation",
        "email": "client@company.com",
        "recommendation": "The agreement should be reviewed by counsel before signing."
      }
    }
  ],
  "result": {
    "sent": true,
    "tool": "send_contract_recommendati

## 7. What the two branches prove

The model only proposed structured arguments. The checkpoint preserved the pending action, and deterministic runtime logic controlled what happened next:

- `rejected` → no execution;
- `executed` → exactly one execution after a named reviewer approved it.

**Reflection:** In a production system, what additional identity, expiry, and audit requirements should be attached to the approval?